# Phase 4 — MuJoCo 3D Visualization

**Project**: Quantum-Inspired Tensor-Network Compression of OpenVLA-7B
**Challenge**: Global Quantum + AI Challenge 2026 — VW Group Enterprise Track
**Phase**: 4 of 7 | **Execution**: Google Colab (GPU + CPU rendering)

---

## What this notebook does

Drives a **dm_control humanoid** figure with action outputs from the
chi=64 compressed OpenVLA-7B model and renders a 3D visualization.

**Pipeline**:
```
OXE sample (image + language)
    → Compressed OpenVLA-7B (chi=64, from Phase 2 Drive checkpoint)
    → 7-DoF delta action [dx, dy, dz, droll, dpitch, dyaw, gripper]
    → action_to_joint_command()  (mujoco_bridge.py)
    → 21-DoF humanoid ctrl  (right arm driven; lower body PD-stabilised)
    → dm_control humanoid physics (CPU)
    → rendered frames → MP4 + GIF
```

**Joint mapping**: OpenVLA's 7-DoF end-effector deltas drive the humanoid's
right arm (shoulder1, shoulder2, elbow — 3 DoF available in standard humanoid).
The lower body is stabilised by a proportional controller holding the standing pose.

## Before running

1. **GPU runtime** (Runtime → Change runtime type → GPU).
2. **Colab Secrets**: `HF_TOKEN` and `GH_TOKEN`.
3. **Phase 2 must have run**: chi=64 checkpoint must exist on Google Drive at
   `MyDrive/vlam_checkpoints/compressed_chi64/cores.pt`.

---

> **License**: OpenVLA-7B weights are subject to the LLaMA-2 Community License
> (non-commercial research use only).

In [ ]:
# ── Cell 1: Install dependencies ──────────────────────────────────────────────
import subprocess, sys, os

def _pip(*args):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *args], check=True)

_pip(
    "transformers==4.40.1",
    "tokenizers==0.19.1",
    "timm==0.9.10",
    "sentencepiece==0.1.99",
)
_pip(
    "accelerate>=0.27.0",
    "bitsandbytes>=0.43.0",
    "dm-control>=1.0.20",
    "mujoco>=3.1.0",
    "av",
    "datasets",
    "imageio[ffmpeg]",
    "Pillow",
    "PyYAML",
    "tqdm",
)

# Set headless rendering backend before importing dm_control
os.environ.setdefault("MUJOCO_GL", "egl")
print(f"MUJOCO_GL={os.environ['MUJOCO_GL']}")
print("pip installs complete.")

In [ ]:
# ── Cell 2: Verify transformers version ───────────────────────────────────────
import transformers
REQUIRED = "4.40.1"
if transformers.__version__ != REQUIRED:
    raise RuntimeError(
        f"transformers {transformers.__version__} installed but need exactly {REQUIRED}.\n"
        "Fix: Runtime -> Disconnect and delete runtime, then re-run Cell 1 first."
    )
print(f"transformers {REQUIRED} confirmed.")

In [ ]:
# ── Cell 3: Clone repo and install vlam_compress ──────────────────────────────
import os, sys, subprocess

REPO_DIR = "/content/vlam"

try:
    from google.colab import userdata
    _gh_token = userdata.get("GH_TOKEN")
    REPO_URL = f"https://{_gh_token}@github.com/quantumadventurer11/vw-quantum-vlam-challenge"
    print("GH_TOKEN loaded.")
except Exception as _e:
    print(f"GH_TOKEN not found ({_e}) — using unauthenticated URL.")
    REPO_URL = "https://github.com/quantumadventurer11/vw-quantum-vlam-challenge"

if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=True)

os.chdir(REPO_DIR)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "."], check=True)
print(f"Working directory: {os.getcwd()}")

In [ ]:
# ── Cell 4: HuggingFace authentication ────────────────────────────────────────
from huggingface_hub import login
try:
    from google.colab import userdata
    login(token=userdata.get("HF_TOKEN"), add_to_git_credential=False)
    print("Logged in via Colab Secrets.")
except Exception as e:
    print(f"Secret not found ({e}). Falling back to interactive login...")
    login()

In [ ]:
# ── Cell 5: Mount Google Drive (Phase 2 checkpoints) ─────────────────────────
from pathlib import Path

try:
    from google.colab import drive
    drive.mount("/content/drive")
    CHECKPOINTS_BASE = Path("/content/drive/MyDrive/vlam_checkpoints")
    print("Google Drive mounted.")
except Exception as _e:
    print(f"Drive mount skipped ({_e}) — using local checkpoints/")
    CHECKPOINTS_BASE = Path("checkpoints")

cores_path = CHECKPOINTS_BASE / "compressed_chi64" / "cores.pt"
if cores_path.exists():
    print(f"Found: {cores_path}  ({cores_path.stat().st_size / 1024**2:.0f} MB)")
else:
    print(f"WARNING: {cores_path} not found. Run Phase 2 first.")

In [ ]:
# ── Cell 6: Imports ────────────────────────────────────────────────────────────
import json
import os
import time
from pathlib import Path

import numpy as np
import torch
import yaml
from datasets import load_dataset
from PIL import Image
from tqdm.auto import tqdm

from vlam_compress.mujoco_bridge import (
    make_env, run_episode, action_to_joint_command,
    encode_video, make_overlay_fn, make_side_by_side,
    load_compressed_model, restore_patches,
    RENDER_FPS, RENDER_WIDTH, RENDER_HEIGHT,
)

RESULTS_DIR = Path("results")
RESULTS_DIR.mkdir(exist_ok=True)

print(f"PyTorch : {torch.__version__}")
print("All imports OK.")

In [ ]:
# ── Cell 7: Verify GPU + dm_control headless rendering ────────────────────────
import os
assert torch.cuda.is_available(), "No GPU — change runtime to GPU and restart."
props = torch.cuda.get_device_properties(0)
print(f"GPU  : {props.name}  ({props.total_memory / 1024**3:.1f} GB VRAM)")
print(f"CUDA : {torch.version.cuda}")

# Test dm_control rendering (will raise if EGL/OSMesa not available)
print("\nTesting dm_control headless render ...")
_env   = make_env(random_state=0)
_env.reset()
_frame = _env.physics.render(height=64, width=64, camera_id=1)
assert _frame.shape == (64, 64, 3), f"Unexpected frame shape: {_frame.shape}"
print(f"Render test OK — frame shape {_frame.shape}, dtype={_frame.dtype}")
del _env, _frame

In [ ]:
# ── Cell 8: Configuration ─────────────────────────────────────────────────────
with open("configs/seeds.yaml") as f:
    cfg = yaml.safe_load(f)

CHI          = 64                  # primary target checkpoint
MODEL_ID     = "openvla/openvla-7b"
UNNORM_KEY   = "bridge_orig"
HF_DATASET   = "lerobot/bridge"
N_STEPS      = 200                 # physics + render cycles (≈ 8 s at 25 fps)
EPISODE_SEED = cfg["seeds"][0]     # 42

print(f"chi         : {CHI}")
print(f"n_steps     : {N_STEPS}  ({N_STEPS / RENDER_FPS:.0f} s at {RENDER_FPS} fps)")
print(f"episode seed: {EPISODE_SEED}")

In [ ]:
# ── Cell 9: Load one bridge episode sample (static scene for all steps) ────────
import json as _json
from huggingface_hub import hf_hub_download

print(f"Loading {HF_DATASET} ...")
hf_ds    = load_dataset(HF_DATASET, split="train", streaming=True)
task_map: dict = {}
try:
    _tf = hf_hub_download(HF_DATASET, filename="meta/tasks.jsonl")
    with open(_tf) as _f:
        for _line in _f:
            _d = _json.loads(_line.strip())
            task_map[_d["task_index"]] = _d["task"]
    print(f"Loaded {len(task_map)} task descriptions.")
except Exception as _e:
    print(f"tasks.jsonl not found ({_e}) — using generic fallback.")

from vlam_compress.mujoco_bridge import _ACTION_MAP  # noqa: confirms import

def hf_item_to_sample(item):
    import av, io as _io
    img = None
    for k in ("observation.image", "observation.images.image_0", "image"):
        if k in item:
            img = item[k]; break
    if isinstance(img, dict):
        _p, _b = img.get("path"), img.get("bytes")
        if _p:
            with av.open(_p) as _c:
                for _f in _c.decode(video=0):
                    img = Image.fromarray(_f.to_ndarray(format="rgb24")); break
        elif _b:
            img = Image.open(_io.BytesIO(_b))
    elif isinstance(img, np.ndarray):
        img = Image.fromarray(img.astype(np.uint8))
    action = np.array(item.get("action", [0]*7), dtype=np.float32).ravel()[:7]
    lang   = item.get("language_instruction") or item.get("task") or ""
    if lang is None or not str(lang).strip():
        tidx = item.get("task_index")
        lang = task_map.get(tidx, "pick up the object") if tidx is not None else "pick up the object"
    if isinstance(lang, (bytes, bytearray)):
        lang = lang.decode()
    return {"image": img, "language": str(lang).strip(), "action_gt": action}

# Grab one fixed sample — reused as the static scene across all rollout steps
_peek     = next(iter(hf_ds.shuffle(seed=EPISODE_SEED, buffer_size=1000).take(1)))
FIXED_SAMPLE = hf_item_to_sample(_peek)
del _peek

print(f"Sample language: '{FIXED_SAMPLE['language']}'")
print(f"Sample image   : {FIXED_SAMPLE['image'].size}  mode={FIXED_SAMPLE['image'].mode}")
get_sample_fn = lambda: FIXED_SAMPLE

In [ ]:
# ── Cell 10: Load compressed model (chi=64) ────────────────────────────────────
# load_compressed_model loads INT8 base + patches layers with chi=64 cores.
model, processor, saved_modules = load_compressed_model(
    chi=CHI,
    checkpoints_base=CHECKPOINTS_BASE,
    model_id=MODEL_ID,
)

n_params = sum(p.numel() for p in model.parameters())
peak_mem = torch.cuda.max_memory_allocated() / 1024**2
print(f"Total params : {n_params / 1e9:.3f}B")
print(f"Peak GPU mem : {peak_mem:.0f} MiB")

In [ ]:
# ── Cell 11: Define predict_fn (GPU inference wrapper) ────────────────────────
def predict_fn(image: Image.Image, language: str) -> np.ndarray:
    inputs = processor(language, image).to("cuda:0")
    with torch.no_grad():
        action = model.predict_action(
            **inputs, unnorm_key=UNNORM_KEY, do_sample=False
        )
    return np.array(action, dtype=np.float32)

# Warm-up
print("Warm-up inference pass ...")
_ = predict_fn(FIXED_SAMPLE["image"], FIXED_SAMPLE["language"])
print("OK")

In [ ]:
# ── Cell 12: Run compressed episode ───────────────────────────────────────────
print(f"Running chi={CHI} compressed episode ({N_STEPS} steps) ...")
t0 = time.perf_counter()

result_compressed = run_episode(
    predict_fn=predict_fn,
    get_sample_fn=get_sample_fn,
    n_steps=N_STEPS,
    random_state=EPISODE_SEED,
    render_width=RENDER_WIDTH,
    render_height=RENDER_HEIGHT,
)

elapsed = time.perf_counter() - t0
print(f"Done — {result_compressed['n_frames']} frames in {elapsed:.0f} s")
print(f"Language: '{result_compressed['language'][:60]}'")

# Arm trajectory summary
arm_arr = np.array(result_compressed["arm_trajectory"])
print(f"Arm range — shoulder1: [{arm_arr[:,0].min():.2f}, {arm_arr[:,0].max():.2f}]")
print(f"            shoulder2: [{arm_arr[:,1].min():.2f}, {arm_arr[:,1].max():.2f}]")
print(f"            elbow    : [{arm_arr[:,2].min():.2f}, {arm_arr[:,2].max():.2f}]")

In [ ]:
# ── Cell 13: Run INT8 baseline episode (same scene) ────────────────────────────
print("Restoring INT8 weights for baseline episode ...")
restore_patches(saved_modules)

print(f"Running INT8 baseline episode ({N_STEPS} steps) ...")
t0 = time.perf_counter()

result_baseline = run_episode(
    predict_fn=predict_fn,      # same fn — now uses restored INT8 weights
    get_sample_fn=get_sample_fn,
    n_steps=N_STEPS,
    random_state=EPISODE_SEED,
    render_width=RENDER_WIDTH,
    render_height=RENDER_HEIGHT,
)

elapsed = time.perf_counter() - t0
print(f"Done — {result_baseline['n_frames']} frames in {elapsed:.0f} s")

In [ ]:
# ── Cell 14: Load compression ratio from Phase 2 stats ────────────────────────
COMPRESSION_RATIO = None
stats_path = RESULTS_DIR / "compression_sweep_stats.json"
if stats_path.exists():
    with open(stats_path) as f:
        _stats = json.load(f)
    COMPRESSION_RATIO = (
        _stats.get("sweep_stats", {})
              .get(str(CHI), {})
              .get("layer_compression_ratio_mean")
    )
    if COMPRESSION_RATIO:
        print(f"Compression ratio from Phase 2: {COMPRESSION_RATIO:.2f}x")
    else:
        print("compression_sweep_stats.json found but chi key missing.")
else:
    print("compression_sweep_stats.json not found — run Phase 2 first.")
    print("Proceeding without compression ratio overlay.")

In [ ]:
# ── Cell 15: Encode individual MP4s ───────────────────────────────────────────
overlay_c = make_overlay_fn(
    label=f"TN chi={CHI}",
    language=result_compressed["language"],
    arm_traj=result_compressed["arm_trajectory"],
    compression_ratio=COMPRESSION_RATIO,
)
overlay_b = make_overlay_fn(
    label="INT8 baseline",
    language=result_baseline["language"],
    arm_traj=result_baseline["arm_trajectory"],
)

out_c = RESULTS_DIR / f"demo_compressed_chi{CHI}.mp4"
out_b = RESULTS_DIR / "demo_baseline_int8.mp4"

encode_video(result_compressed["frames"], out_c, fps=RENDER_FPS, overlay_fn=overlay_c)
encode_video(result_baseline["frames"],   out_b, fps=RENDER_FPS, overlay_fn=overlay_b)

print(f"Written: {out_c}  (+ .gif)")
print(f"Written: {out_b}  (+ .gif)")

# Verification: duration check
duration_s = result_compressed["n_frames"] / RENDER_FPS
print(f"\nVideo duration: {duration_s:.1f} s  (requirement: >= 5 s)")
assert duration_s >= 5.0, f"Video too short: {duration_s:.1f} s < 5 s"

In [ ]:
# ── Cell 16: Encode side-by-side comparison ────────────────────────────────────
sbs_frames = make_side_by_side(
    frames_a=result_baseline["frames"],
    frames_b=result_compressed["frames"],
    label_a="INT8 baseline",
    label_b=f"TN chi={CHI}",
)

out_sbs = RESULTS_DIR / f"demo_side_by_side_chi{CHI}.mp4"
encode_video(sbs_frames, out_sbs, fps=RENDER_FPS)
print(f"Written: {out_sbs}  (+ .gif)")
print(f"Frame size: {sbs_frames[0].shape}  ({sbs_frames[0].shape[1]}px wide)")

In [ ]:
# ── Cell 17: Arm trajectory plot ──────────────────────────────────────────────
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

arm_c = np.array(result_compressed["arm_trajectory"])
arm_b = np.array(result_baseline["arm_trajectory"])
steps = np.arange(len(arm_c))

fig, axes = plt.subplots(3, 1, figsize=(10, 6), sharex=True)
joint_names = ["shoulder1", "shoulder2", "elbow"]
colors_c = ["#3a86ff", "#8338ec", "#ff006e"]
colors_b = ["#adb5bd", "#adb5bd", "#adb5bd"]

for j, (ax, name) in enumerate(zip(axes, joint_names)):
    ax.plot(steps, arm_c[:, j], color=colors_c[j], label=f"TN chi={CHI}", linewidth=1.5)
    ax.plot(steps, arm_b[:, j], color=colors_b[j], label="INT8 baseline",
            linewidth=1.0, linestyle="--", alpha=0.7)
    ax.set_ylabel(f"{name}\n(ctrl)")
    ax.legend(loc="upper right", fontsize=8)
    ax.grid(True, alpha=0.3)

axes[-1].set_xlabel("Step")
fig.suptitle(f"Right arm trajectory — chi={CHI} vs INT8 baseline", fontweight="bold")
plt.tight_layout()

plot_path = RESULTS_DIR / f"arm_trajectory_chi{CHI}.png"
plt.savefig(str(plot_path), dpi=120, bbox_inches="tight")
plt.show()
print(f"Saved: {plot_path}")

In [ ]:
# ── Cell 18: Verification checklist ───────────────────────────────────────────
print("── Verification ──────────────────────────────────────────────────────────")

# 1. Duration >= 5 s
dur = result_compressed["n_frames"] / RENDER_FPS
ok_dur = dur >= 5.0
print(f"[{'OK  ' if ok_dur else 'FAIL'}] Video duration >= 5 s  : {dur:.1f} s")

# 2. Non-trivial arm motion (arm range > 0.05 in at least one joint)
arm_c = np.array(result_compressed["arm_trajectory"])
arm_range = arm_c.max(axis=0) - arm_c.min(axis=0)
ok_motion = arm_range.max() > 0.05
print(f"[{'OK  ' if ok_motion else 'FAIL'}] Non-trivial arm motion : max range={arm_range.max():.3f}")

# 3. Baseline vs compressed arm trajectories differ
arm_b = np.array(result_baseline["arm_trajectory"])
n = min(len(arm_c), len(arm_b))
diff = np.abs(arm_c[:n] - arm_b[:n]).mean()
ok_diff = diff > 1e-4
print(f"[{'OK  ' if ok_diff else 'FAIL'}] Compressed != baseline  : mean |diff|={diff:.5f}")

# 4. Output files exist
out_c   = RESULTS_DIR / f"demo_compressed_chi{CHI}.mp4"
out_sbs = RESULTS_DIR / f"demo_side_by_side_chi{CHI}.mp4"
ok_c    = out_c.exists()
ok_sbs  = out_sbs.exists()
print(f"[{'OK  ' if ok_c else 'FAIL'}] demo_compressed_chi{CHI}.mp4 exists")
print(f"[{'OK  ' if ok_sbs else 'FAIL'}] demo_side_by_side_chi{CHI}.mp4 exists")

print("──────────────────────────────────────────────────────────────────────────")

In [ ]:
# ── Cell 19: Download results from Colab ──────────────────────────────────────
from google.colab import files

to_download = [
    RESULTS_DIR / f"demo_compressed_chi{CHI}.mp4",
    RESULTS_DIR / f"demo_compressed_chi{CHI}.gif",
    RESULTS_DIR / f"demo_side_by_side_chi{CHI}.mp4",
    RESULTS_DIR / f"demo_side_by_side_chi{CHI}.gif",
    RESULTS_DIR / f"arm_trajectory_chi{CHI}.png",
]

for p in to_download:
    if p.exists():
        files.download(str(p))
        print(f"Downloading {p.name} ({p.stat().st_size / 1024**2:.1f} MB)")
    else:
        print(f"  skip (not found): {p.name}")

print("\nCommit to repo (use git add -f for gitignored extensions):")
print(f"  git add -f results/demo_compressed_chi{CHI}.mp4 results/demo_compressed_chi{CHI}.gif")
print(f"  git add -f results/demo_side_by_side_chi{CHI}.mp4 results/demo_side_by_side_chi{CHI}.gif")
print(f"  git add results/arm_trajectory_chi{CHI}.png")
print(f"  git commit -m '[phase4] add MuJoCo visualization chi={CHI}'")

## Results Summary

After running all cells the following artefacts are produced:

| File | Contents |
|---|---|
| `results/demo_compressed_chi64.mp4` | Humanoid driven by chi=64 model (+ .gif) |
| `results/demo_baseline_int8.mp4` | Humanoid driven by INT8 baseline (+ .gif) |
| `results/demo_side_by_side_chi64.mp4` | Side-by-side comparison (+ .gif) |
| `results/arm_trajectory_chi64.png` | Right arm joint state plot |

### Challenge sections satisfied

- **User requirement**: 3D visualization of humanoid executing a task driven by
  the compressed model's outputs, using MuJoCo ✓
- **§5.2** Functional QI component demonstrated in a downstream task ✓
- **§5.5** Qualitative comparison: compressed vs. baseline arm trajectory ✓